# 🐍 Day 33 · Python Basics — Bytes, Images, Hashing & Watching Folders (Production RAG)

> **Goal of this notebook:** This pipeline turns PDF pages into **images**, sends them as **base64**, remembers processed files with **hashes** and **watches** a folder for new PDFs.

**How to use it**
1. Click on a grey code cell and press **Shift + Enter** to run it.
2. Read the comments (lines that start with `#`) — they explain every step.
3. Change the values and run the cell again. Breaking things is how you learn! 🙂

📦 **First** do *Step 1 - Install today's packages* in [`00_START_HERE.md`](00_START_HERE.md).

✅ No API key needed for this notebook.

### 📋 What you will learn
1. Creating an image with Pillow
2. Saving an image to memory with `BytesIO`
3. base64 for sending images to an API
4. File fingerprints with `hashlib`
5. Remembering processed files in JSON
6. 🔗 A simple folder watcher

## 1. Create an image with Pillow (`PIL`)
`document_processor.py` turns every PDF page into an image like this one.

In [ ]:
from PIL import Image, ImageDraw

page = Image.new("RGB", (400, 200), color="white")       # width x height
draw = ImageDraw.Draw(page)
draw.rectangle([20, 20, 380, 60], outline="navy", width=3)
draw.text((30, 32), "JPMorgan 2023 - Page 1", fill="navy")
draw.text((30, 100), "Net income: $49.6B", fill="black")
page                     # the last line of a cell is displayed — you will see the image

## 2. Save the image to memory
`BytesIO` is a pretend file that lives in memory — no file on disk needed.

In [ ]:
from io import BytesIO

buffer = BytesIO()
page.save(buffer, format="PNG")
png_bytes = buffer.getvalue()
print(len(png_bytes), "bytes, starts with", png_bytes[:8])

## 3. base64 for the API
The Gemini embedding API receives images as base64 **text** inside JSON.

In [ ]:
import base64

b64 = base64.b64encode(png_bytes).decode("utf-8")
print(b64[:60] + "...")
payload = {"mime_type": "image/png", "data": b64}      # roughly what gets sent
print(list(payload))

## 4. File fingerprints with `hashlib`
A **hash** turns any data into a short fingerprint. Same content → same hash. One changed byte → a totally different hash.
That's how the watcher knows a PDF was already processed, even if it was renamed.

In [ ]:
import hashlib

print(hashlib.md5(b"annual report 2023").hexdigest())
print(hashlib.md5(b"annual report 2023").hexdigest())   # identical
print(hashlib.md5(b"annual report 2024").hexdigest())   # completely different

def file_hash(path):
    return hashlib.md5(open(path, "rb").read()).hexdigest()

## 5. 🔗 A folder watcher
`watcher.py` checks the `data/` folder. When a **new** PDF appears, it processes it and saves its hash in `.processed_files.json`.

In [ ]:
import json
from pathlib import Path

inbox = Path("demo_inbox"); inbox.mkdir(exist_ok=True)
memory_file = inbox / ".processed.json"

def load_processed():
    return set(json.loads(memory_file.read_text(encoding="utf-8"))) if memory_file.exists() else set()

def save_processed(hashes):
    memory_file.write_text(json.dumps(sorted(hashes)), encoding="utf-8")

def check_folder():
    processed = load_processed()
    for pdf in sorted(inbox.glob("*.pdf")):
        h = file_hash(pdf)
        if h in processed:
            print(f"⏭️  skip {pdf.name} (already processed)")
            continue
        print(f"⚙️  processing {pdf.name} ... (pages -> images -> embeddings -> Qdrant)")
        processed.add(h)
    save_processed(processed)

(inbox / "report_A.pdf").write_bytes(b"fake pdf A")
check_folder()
print("--- a new file arrives ---")
(inbox / "report_B.pdf").write_bytes(b"fake pdf B")
(inbox / "copy_of_A.pdf").write_bytes(b"fake pdf A")      # same content, new name
check_folder()

In [ ]:
import shutil
shutil.rmtree(inbox)
print("demo_inbox deleted ✅")

---
## 🏋️ Practice time
Try these on your own in the empty cell below:

1. Draw your name on a new 300×100 image.
2. Compute the md5 hash of the text `"hello"` (remember to use bytes: `b"hello"`).
3. What would happen in `check_folder` if `.processed.json` were deleted?

In [ ]:
# ✍️ Write your answers here

<details>
<summary>👀 Click here to see the solutions (try first!)</summary>

```python
img = Image.new("RGB", (300, 100), "white")
ImageDraw.Draw(img).text((20, 40), "Priya", fill="black")
img

print(hashlib.md5(b"hello").hexdigest())
# Without the JSON memory every PDF would be processed again (wasting time and API money).
```
</details>

## ✅ Summary

| Concept | Example |
|---|---|
| New image | `Image.new('RGB', (400, 200), 'white')` |
| Draw text | `ImageDraw.Draw(img).text((x, y), 'hi')` |
| Image -> bytes | `img.save(BytesIO(), format='PNG')` |
| base64 text | `base64.b64encode(b).decode()` |
| Fingerprint | `hashlib.md5(data).hexdigest()` |
| Read file bytes | `open(p, 'rb').read()` |

## ➡️ What's next?
You now know the Python used in **Day_33_Production_RAG_Pipeline**.

Open [`00_START_HERE.md`](00_START_HERE.md) and continue with **lesson 1**. 🚀